Keras

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [2]:
# 1. Cargar el dataset MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# 2. Normalización
x_train = x_train / 255.0
x_test = x_test / 255.0

In [4]:
# 3. Definición del modelo CNN
model = Sequential([
    tf.keras.layers.Reshape((28,28,1), input_shape=(28,28)),  # Asegura forma (alto, ancho, canales)
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')  # 10 clases de salida
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [5]:
# 4. Compilación del modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # Porque las etiquetas no están one-hot encoded
    metrics=['accuracy']
)

In [6]:
# 5. Entrenamiento del modelo
model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_test, y_test))

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 54s 56ms/step - accuracy: 0.8718 - loss: 0.4305 - val_accuracy: 0.9830 - val_loss: 0.0527
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 85s 59ms/step - accuracy: 0.9829 - loss: 0.0552 - val_accuracy: 0.9878 - val_loss: 0.0396
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 79s 56ms/step - accuracy: 0.9886 - loss: 0.0370 - val_accuracy: 0.9871 - val_loss: 0.0345
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 81s 55ms/step - accuracy: 0.9907 - loss: 0.0291 - val_accuracy: 0.9888 - val_loss: 0.0325
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 52s 56ms/step - accuracy: 0.9926 - loss: 0.0225 - val_accuracy: 0.9908 - val_loss: 0.0287
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 56ms/step - accuracy: 0.9950 - loss: 0.0155 - val_accuracy: 0.9904 - val_loss: 0.0308
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 80s 54ms/step - accuracy: 0.9956 - loss: 0.0120 - val_accuracy: 0.9902 - val_loss: 0.0318
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 54ms/step - accuracy: 0.9964 - loss: 0.0110 - 

In [7]:
# 6. Evaluación final (opcional)
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'\nPrecisión en el conjunto de prueba: {test_acc*100:.2f}%')

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9885 - loss: 0.0435

Precisión en el conjunto de prueba: 99.12%


PyTorch

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [9]:
# 1. Transformaciones
transform = transforms.ToTensor()

In [10]:
# 2. Dataset y DataLoader
train_dataset = datasets.MNIST(root='.', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='.', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 133MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 26.1MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 59.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.94MB/s]


In [11]:
# 3. Definición de la red
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)   # Entrada: 28x28x1 -> Salida: 28x28x32
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)  # Entrada: 14x14x32 -> Salida: 14x14x64
        self.pool = nn.MaxPool2d(2, 2)                # Reducción: /2 en ancho y alto
        self.fc1 = nn.Linear(64 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))      # 28x28 -> 14x14
        x = self.pool(torch.relu(self.conv2(x)))      # 14x14 -> 7x7
        x = x.view(-1, 64 * 7 * 7)                     # Aplanar
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [12]:
# 4. Inicializar modelo
model = CNN()

In [13]:
# 5. Función de pérdida y optimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
# 6. Entrenamiento
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Época {epoch+1}/{epochs}, Pérdida Promedio: {running_loss/len(train_loader):.4f}")

Época 1/5, Pérdida Promedio: 0.2082
Época 2/5, Pérdida Promedio: 0.0539
Época 3/5, Pérdida Promedio: 0.0374
Época 4/5, Pérdida Promedio: 0.0283
Época 5/5, Pérdida Promedio: 0.0217


In [15]:
# 7. Evaluación en test
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

In [16]:
print(f"Precisión en el conjunto de prueba: {100 * correct / total:.2f}%")

Precisión en el conjunto de prueba: 99.07%
